# The paper's protocol - does the keep-only result survive disease-centric ranking?

Every number in this study so far uses `evaluate_fb`, TxGNN's **edge-level** evaluation, which samples
**one negative per positive**. Average precision on a balanced set means a random scorer gets exactly
0.500, and that is why every result here is quoted against a 0.500 floor.

That is not the paper's headline protocol. The paper ranks **every candidate drug for each held-out
disease** and reports recall and enrichment at k. That protocol is far harsher: the positive rate per
disease is a fraction of a percent rather than 50%, so a model can look strong at edge level and weak
at ranking. A reviewer will say so, and they would be right to.

The library already implements it: **`disease_centric_evaluation`** (`utils.py:1132`) scores all drugs
for every test disease and returns per-disease `AUROC`, `AUPRC`, `Recall@{1%,5%,10%,10,50,100}`,
`MRR@k`, `AP@k`, `Hits@k` and `Enrichment@k` for each of the three drug-disease relations. This notebook
uses that function unmodified - no metric is invented here.

## What is tested

The claim most in need of protocol validation is the keep-only headline, because it is the paper
draft's strongest structural result:

> Reducing the graph to 9 edge types of 50 - 3.9% of the training edges - retains ~80% of the
> above-chance indication margin.

So two arms, evaluated with **both** protocols from the same trained model:

| arm | relations kept | edge types |
|---|---|---|
| `full_kg` | all | 50 |
| `only_disease_relations` | task labels + `disease_disease` + `disease_protein` | 9 |

Nothing is imported: the existing grids never ran `disease_centric_evaluation`, and no checkpoints were
saved, so both arms must be retrained here. That is the cost of this experiment and there is no way
around it.

`SEEDS` therefore defaults to **10**, not 20. Ten paired seeds already gave CIs of about +/-0.03 on the
keep-only delta, which is far tighter than the effect being tested, and the grid is seed-major and
resumable so extending to 20 later costs nothing but time.

## Two engineering notes

- **`SAVE_MODELS` defaults to `True` here**, unlike every other grid. `disease_centric_evaluation` is a
  long, rarely-exercised code path reading `kg.csv` from disk; if it raises after a 25-minute training
  run, the checkpoint means the ranking can be retried in seconds with `rerank(arm, seed)` instead of
  retraining. Ranking failures are recorded in `run_meta.json` as `ranking_error` and excluded from the
  ranking aggregation, so one bad run does not block the grid.
- **`disease_centric_evaluation` mutates its `df` argument in place** (`df['x_id'] = df.x_id.apply(...)`,
  `utils.py:1151`). It is passed `d.df.copy()` so the cached split frames shared across arms are never
  modified - otherwise arm 2 would silently train on a frame arm 1 had rewritten.

---
## 1 - Environment

In [ ]:
import os, sys, json, time, gc, random, shutil, inspect, platform, traceback, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')


def _find_repo_root(start=None):
    p = Path(start or os.getcwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / 'pyg_implementation' / 'txgnn').is_dir():
            return cand
    raise RuntimeError('Could not find pyg_implementation/txgnn above ' + str(p))


REPO_ROOT = _find_repo_root()
PYG_ROOT  = REPO_ROOT / 'pyg_implementation'
sys.path.insert(0, str(PYG_ROOT))

import torch
import torch.nn as nn

import txgnn
from txgnn import TxData, TxGNN
from txgnn.utils import (evaluate_fb, create_pyg_graph, get_n_params,
                         disease_centric_evaluation)

assert 'pyg_implementation' in txgnn.__file__, (
    'Wrong txgnn package on sys.path (%s). Restart the kernel.' % txgnn.__file__)

_MI_PARAMS = set(inspect.signature(TxGNN.model_initialize).parameters)
assert 'use_decision_network' not in _MI_PARAMS, (
    'model_initialize accepts use_decision_network again - revisit this notebook.')

# the ranking evaluation reads kg.csv itself, from the folder handed to TxData
_DCE_PARAMS = set(inspect.signature(disease_centric_evaluation).parameters)
for _p in ('df', 'df_train', 'df_valid', 'df_test', 'data_path', 'G', 'model', 'device'):
    assert _p in _DCE_PARAMS, 'disease_centric_evaluation lost parameter %r' % _p

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


print('repo  :', REPO_ROOT)
print('txgnn :', txgnn.__file__)
print('torch :', torch.__version__, '| device:', DEVICE,
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else platform.processor())

---
## 2 - Configuration

In [ ]:
SPLIT   = 'complex_disease'
SEEDS   = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]      # 10, not 20 - see cell 0. Extend freely.
PROFILE = 'full'

MODEL_CFG = dict(
    n_hid       = 100,
    n_inp       = 100,
    n_out       = 100,
    proto       = True,
    proto_num   = 3,
    attention   = False,
    sim_measure = 'all_nodes_profile',
    agg_measure = 'rarity',
)

PRETRAIN_CFG = dict(n_epoch=1,    learning_rate=1e-3, batch_size=1024, train_print_per_n=500)
FINETUNE_CFG = dict(n_epoch=1000, learning_rate=5e-4, train_print_per_n=100, valid_per_n=20)

DD_REL     = ['contraindication', 'indication', 'off-label use']
DD_REV_REL = ['rev_' + r for r in DD_REL]
DD_ETYPES  = [('drug', r, 'disease') for r in DD_REL] + [('disease', r, 'drug') for r in DD_REV_REL]

SIG_ETYPES = ['disease_disease', 'rev_disease_protein']

DATA_FOLDER = REPO_ROOT / 'data'      # must contain kg.csv: both TxData and the ranking eval read it
ABL_ROOT    = REPO_ROOT / 'results' / 'ablation_full_ranking'
MODEL_ROOT  = REPO_ROOT / 'saved_models' / 'ablation_full_ranking'
ABL_ROOT.mkdir(parents=True, exist_ok=True)

REF_ARM = 'full_kg'

# the per-disease columns worth keeping; everything else in the returned frame is a nested dict
RANKING_HEADLINE = ['Recall@1%', 'Recall@5%', 'Recall@10%', 'Recall@10', 'Recall@50',
                    'Enrichment@5%', 'Enrichment@10%', 'MRR@10', 'MRR@50',
                    'AP@10', 'AP@50', 'AUROC', 'AUPRC']
RANKING_DROP = ['Ranked List', 'Prediction', 'Labels', 'ID', 'Name']   # huge / non-numeric

# --- switches ------------------------------------------------------------------------------------
RUN_GRID          = True
SAVE_MODELS       = True    # True on purpose here - see cell 0
RUN_RANKING       = True    # False trains and does edge-level only
STOP_AFTER_N_RUNS = None

print('split  :', SPLIT)
print('seeds  : %d -> %s' % (len(SEEDS), SEEDS))
print('results:', ABL_ROOT.relative_to(REPO_ROOT))
assert (DATA_FOLDER / 'kg.csv').exists() or not RUN_RANKING, (
    'disease_centric_evaluation reads %s, which does not exist. Point DATA_FOLDER at the folder '
    'holding kg.csv, or set RUN_RANKING = False.' % (DATA_FOLDER / 'kg.csv'))

---
## 3 - Arms and mechanics

`KeptData` is the same stand-in used by `ablation_keep_disease_only.ipynb`: `base['df']` is the
untouched full KG, so `create_pyg_graph(df_train, df)` keeps every node type and every node index
identical across arms and only edges disappear.

In [ ]:
DD_RELATIONS        = ['indication', 'contraindication', 'off-label use']
SIGNATURE_RELATIONS = ['disease_disease', 'disease_protein']

ARMS = [
    {'arm': 'full_kg', 'keep': None, 'expect_etypes': 50,
     'note': 'reference: the unmodified KG'},
    {'arm': 'only_disease_relations', 'keep': DD_RELATIONS + SIGNATURE_RELATIONS,
     'expect_etypes': 9, 'note': 'the keep-only headline, re-tested under the ranking protocol'},
]
ARMS_BY_NAME = {a['arm']: a for a in ARMS}
assert REF_ARM in ARMS_BY_NAME


def expand_relations(names):
    out = set()
    for r in names:
        out |= {r, 'rev_' + r}
    return out


def keep_relations(df, keep):
    if keep is None:
        return df.reset_index(drop=True)
    return df[df.relation.isin(expand_relations(keep))].reset_index(drop=True)


class KeptData:
    def __init__(self, base, keep, data_folder, split):
        self.df = base['df']                   # untouched: fixes node counts across arms
        self.df_train = keep_relations(base['df_train'], keep)
        self.df_valid = keep_relations(base['df_valid'], keep)
        self.df_test  = keep_relations(base['df_test'],  keep)
        self.G = create_pyg_graph(self.df_train, self.df)
        self.data_folder = str(data_folder)
        self.split = split
        self.disease_eval_idx = None
        self.no_kg = False


_SEED_CACHE = {}


def get_seed_data(seed):
    if seed in _SEED_CACHE:
        return _SEED_CACHE[seed]
    _SEED_CACHE.clear()
    gc.collect()
    print('[data] loading split seed %d ...' % seed)
    t0 = time.time()
    set_all_seeds(seed)
    d = TxData(data_folder_path=str(DATA_FOLDER))
    d.prepare_split(split=SPLIT, seed=seed, no_kg=False)
    base = {'df': d.df, 'df_train': d.df_train, 'df_valid': d.df_valid, 'df_test': d.df_test}
    del d
    gc.collect()
    _SEED_CACHE[seed] = base
    print('[data] seed %d ready in %.1f s | train edges %d'
          % (seed, time.time() - t0, len(base['df_train'])))
    return base


def run_dir(arm, seed):
    return ABL_ROOT / ('%s_seed%d' % (arm, seed))


def is_done(arm, seed):
    d = run_dir(arm, seed)
    return (d / 'edge_level_test_metrics.csv').exists() and (d / 'run_meta.json').exists()


def encoder_params(model):
    return (sum(p.numel() for p in model.layer1.parameters())
            + sum(p.numel() for p in model.layer2.parameters()))

### 3.1 - The ranking evaluation wrapper

`disease_centric_evaluation` returns one DataFrame per reverse relation, indexed by disease, with the
per-disease metrics. Two things need care:

1. **Diseases with no positives are flagged, not dropped.** `AUROC` and `AUPRC` are set to `-1`
   (`utils.py:1270`) when a disease has no positive drug in the evaluated set. Averaging without
   filtering those would silently drag every mean down. `summarise_ranking` drops them per metric.
2. **The frame carries three nested-dict columns** (`Ranked List`, `Prediction`, `Labels`) that are
   megabytes per disease. They are dropped before anything is written to disk.

In [ ]:
def run_ranking(m, d, verbose=False):
    '''disease_centric_evaluation over all test diseases. Returns {rel_type: per-disease DataFrame}.'''
    return disease_centric_evaluation(
        df=d.df.copy(),            # the function rewrites df['x_id'] in place - never pass the original
        df_train=d.df_train, df_valid=d.df_valid, df_test=d.df_test,
        data_path=str(DATA_FOLDER),
        G=m.G, model=m.best_model, device=DEVICE,
        disease_ids=None, relation=None,
        weight_bias_track=False, show_plot=False, verbose=verbose, return_raw=False)


def summarise_ranking(per_rel):
    '''{rel: per-disease frame} -> tidy (relation, metric, mean, std, n) frame.

    `-1` is the "no positives for this disease" sentinel for AUROC/AUPRC and is excluded per metric.
    '''
    rows = []
    for rel, frame in per_rel.items():
        f = frame.drop(columns=[c for c in RANKING_DROP if c in frame.columns], errors='ignore')
        for col in f.columns:
            s = pd.to_numeric(f[col], errors='coerce')
            s = s[s.notna() & (s != -1)]
            if len(s):
                rows.append({'relation': rel, 'metric': col, 'mean': float(s.mean()),
                             'std': float(s.std(ddof=1)) if len(s) > 1 else np.nan,
                             'n_diseases': int(len(s))})
    return pd.DataFrame(rows)


def write_ranking(rdir, per_rel):
    '''Per-disease frames (slimmed) + the tidy summary. Returns the summary frame.'''
    for rel, frame in per_rel.items():
        f = frame.drop(columns=[c for c in RANKING_DROP if c in frame.columns], errors='ignore')
        safe = rel.replace(' ', '_').replace('/', '_')
        f.to_csv(rdir / ('disease_centric_per_disease_%s.csv' % safe))
    summ = summarise_ranking(per_rel)
    summ.to_csv(rdir / 'disease_centric_summary.csv', index=False)
    return summ


def rerank(arm_name, seed):
    '''Redo ONLY the ranking evaluation from a saved checkpoint. Seconds, not a retrain.

    For when disease_centric_evaluation failed on a run whose model was saved.
    '''
    arm  = ARMS_BY_NAME[arm_name]
    rdir = run_dir(arm_name, seed)
    mdir = MODEL_ROOT / ('%s_seed%d' % (arm_name, seed))
    if not (mdir / 'model.pt').exists():
        raise FileNotFoundError('no checkpoint at %s - this run has to be retrained' % mdir)

    base = get_seed_data(seed)
    set_all_seeds(seed)
    d = KeptData(base, arm['keep'], DATA_FOLDER, SPLIT)
    m = TxGNN(data=d, weight_bias_track=False, exp_name='rerank', device=DEVICE)
    m.load_pretrained(str(mdir))

    t0 = time.time()
    summ = write_ranking(rdir, run_ranking(m, d))
    print('[rerank] %s_seed%d ranking redone in %.1f min' % (arm_name, seed, (time.time() - t0) / 60))

    meta_p = rdir / 'run_meta.json'
    if meta_p.exists():
        meta = json.loads(meta_p.read_text())
        meta.pop('ranking_error', None)
        meta['ranking_seconds'] = time.time() - t0
        meta['reranked_at'] = time.strftime('%Y-%m-%dT%H:%M:%S')
        meta_p.write_text(json.dumps(meta, indent=2, default=str))

    del m, d
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summ


# --- plan -----------------------------------------------------------------------------------------
SWAP_ROOT = REPO_ROOT / 'results' / 'ablation_encoder_swap'
KEEP_ROOT = REPO_ROOT / 'results' / 'ablation_keep_disease_only'


def _mean_minutes(root, arm):
    ts = []
    for p in sorted(Path(root).glob('%s_seed*/run_meta.json' % arm)):
        r = json.loads(p.read_text())
        if r.get('pretrain_seconds') and r.get('finetune_seconds'):
            ts.append((r['pretrain_seconds'] + r['finetune_seconds']) / 60)
    return float(np.mean(ts)) if ts else None


EST = {'full_kg': _mean_minutes(SWAP_ROOT, 'rgcn') or 24.6,
       'only_disease_relations': _mean_minutes(KEEP_ROOT, 'only_disease_relations') or 3.0}

rows, total = [], 0.0
for a in ARMS:
    done = [s for s in SEEDS if is_done(a['arm'], s)]
    todo = len(SEEDS) - len(done)
    est = todo * EST[a['arm']]
    total += est
    rows.append({'arm': a['arm'], 'edge types': a['expect_etypes'],
                 'train min/run': round(EST[a['arm']], 1),
                 'done': len(done), 'todo': todo, 'est train GPU-h': round(est / 60, 2)})
display(pd.DataFrame(rows).style.hide(axis='index'))

RUN_KEYS = [(a['arm'], s) for s in SEEDS for a in ARMS]
pending = [k for k in RUN_KEYS if not is_done(*k)]
print('\n%d run_keys, %d done, %d pending | ~%.1f GPU-hours of TRAINING' % (
    len(RUN_KEYS), len(RUN_KEYS) - len(pending), len(pending), total / 60))
print('plus disease_centric_evaluation per run - unmeasured here; the first run prints its cost.')
for arm, seed in RUN_KEYS:
    print('  [%s] %s_seed%d' % ('x' if is_done(arm, seed) else ' ', arm, seed))

---
## 4 - The grid

Order of operations per run, chosen so nothing expensive is lost to a late failure:

1. train, 2. edge-level metrics -> CSV, 3. save the checkpoint, 4. ranking -> CSVs,
5. `run_meta.json` last.

A ranking failure is caught, printed with its traceback, and recorded as `ranking_error`; the run still
completes so the grid continues, and `rerank(arm, seed)` fixes it from the checkpoint afterwards.

In [ ]:
def run_one(arm_name, seed, verbose=True):
    arm  = ARMS_BY_NAME[arm_name]
    rdir = run_dir(arm_name, seed)
    el_csv, meta_json = rdir / 'edge_level_test_metrics.csv', rdir / 'run_meta.json'

    if el_csv.exists() and meta_json.exists():
        if verbose:
            print('[skip] %s_seed%d -> cached' % (arm_name, seed))
        return pd.read_csv(el_csv)

    print('\n' + '#' * 78)
    print('# %s   seed %d   keep=%s   expect %d edge types'
          % (arm_name, seed, arm['keep'], arm['expect_etypes']))
    print('#' * 78)

    base = get_seed_data(seed)
    set_all_seeds(seed)
    d = KeptData(base, arm['keep'], DATA_FOLDER, SPLIT)
    etypes_after = sorted('%s|%s|%s' % et for et in d.G.edge_types)
    n_et = len(d.G.edge_types)
    rel_names = {et[1] for et in d.G.edge_types}

    assert n_et == arm['expect_etypes'], \
        '%s: %d edge types, expected %d' % (arm_name, n_et, arm['expect_etypes'])
    want_nodes = set(d.df.x_type) | set(d.df.y_type)
    assert set(d.G.node_types) == want_nodes, 'node types changed'
    sig_sources = [r for r in SIG_ETYPES if r in rel_names]
    assert len(sig_sources) == 2, 'both arms must keep both signature sources, got %s' % sig_sources
    print('[graph] train edges %d of %d (%.2f%%) | %d edge types'
          % (len(d.df_train), len(base['df_train']),
             100.0 * len(d.df_train) / len(base['df_train']), n_et))

    set_all_seeds(seed)
    m = TxGNN(data=d, weight_bias_track=False, exp_name='%s_seed%d' % (arm_name, seed), device=DEVICE)
    m.model_initialize(**MODEL_CFG)
    print('[model] encoder params %s | total %s'
          % (format(encoder_params(m.model), ','), format(get_n_params(m.model), ',')))

    t0 = time.time()
    set_all_seeds(seed)
    m.pretrain(**PRETRAIN_CFG)
    pre_s = time.time() - t0

    t0 = time.time()
    set_all_seeds(seed)
    m.finetune(**FINETUNE_CFG)
    fin_s = time.time() - t0

    # ---- 2. edge-level, the metric every other grid uses ----
    (auroc_rel, auprc_rel, micro_auroc, micro_auprc, macro_auroc, macro_auprc), test_loss = \
        evaluate_fb(m.best_model, m.g_test_pos, m.g_test_neg, m.G, m.dd_etypes, DEVICE, mode='test')

    el = pd.DataFrame(
        [('%s --%s--> %s' % et, auroc_rel.get(et, np.nan), auprc_rel.get(et, np.nan))
         for et in DD_ETYPES],
        columns=['relation', 'AUROC', 'AUPRC'])
    el.loc[len(el)] = ['MICRO (drug-disease)', micro_auroc, micro_auprc]
    el.loc[len(el)] = ['MACRO (drug-disease)',
                       float(np.nanmean([auroc_rel.get(et, np.nan) for et in DD_ETYPES])),
                       float(np.nanmean([auprc_rel.get(et, np.nan) for et in DD_ETYPES]))]
    el.loc[len(el)] = ['MACRO (all relations)', macro_auroc, macro_auprc]
    rdir.mkdir(parents=True, exist_ok=True)
    el.to_csv(el_csv, index=False)

    # ---- 3. checkpoint, so a ranking failure is not a lost training run ----
    if SAVE_MODELS:
        mdir = MODEL_ROOT / ('%s_seed%d' % (arm_name, seed))
        mdir.mkdir(parents=True, exist_ok=True)
        m.save_model(str(mdir))
        print('[save] %s' % mdir.relative_to(REPO_ROOT))

    # ---- 4. the paper's protocol ----
    rank_s, ranking_error, rank_summary = None, None, None
    if RUN_RANKING:
        print('[ranking] disease_centric_evaluation over all test diseases ...')
        t0 = time.time()
        try:
            rank_summary = write_ranking(rdir, run_ranking(m, d))
            rank_s = time.time() - t0
            print('[ranking] done in %.1f min' % (rank_s / 60))
            head = rank_summary[rank_summary.metric.isin(RANKING_HEADLINE)]
            for rel in sorted(head.relation.unique()):
                sub = head[head.relation == rel].set_index('metric')['mean']
                print('   %-22s %s' % (rel, '  '.join(
                    '%s=%.4f' % (k, sub[k]) for k in RANKING_HEADLINE if k in sub.index)))
        except Exception as e:
            ranking_error = '%s: %s' % (type(e).__name__, e)
            rank_s = time.time() - t0
            print('[ranking] FAILED after %.1f min - %s' % (rank_s / 60, ranking_error))
            traceback.print_exc()
            print('[ranking] the run still completes; fix with rerank(%r, %d)' % (arm_name, seed))

    meta = {
        'run_key': '%s_seed%d' % (arm_name, seed),
        'arm': arm_name, 'seed': seed, 'split': SPLIT, 'profile': PROFILE,
        'experiment': 'full_ranking',
        'note': arm['note'],
        'protocols': ['edge_level_fb'] + ([] if ranking_error or not RUN_RANKING
                                          else ['disease_centric']),
        'ranking_error': ranking_error,
        'ranking_seconds': rank_s,
        'kept_relations_requested': arm['keep'],
        'n_edge_types_after': n_et,
        'edge_types_after': etypes_after,
        'signature_sources_present': sig_sources,
        'encoder_params': encoder_params(m.model),
        'n_params': get_n_params(m.model),
        'proto': MODEL_CFG['proto'],
        'signature_disease_etypes': SIG_ETYPES,
        'dropped_edges_total': len(base['df_train']) - len(d.df_train),
        'train_edges_before': len(base['df_train']),
        'train_edges_after': len(d.df_train),
        'model_config': m.config,
        'pretrain': PRETRAIN_CFG, 'finetune': FINETUNE_CFG,
        'pretrain_seconds': pre_s, 'finetune_seconds': fin_s,
        'test_bce_loss': float(test_loss),
        'torch': torch.__version__,
        'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'finished_at': time.strftime('%Y-%m-%dT%H:%M:%S'),
    }
    with open(meta_json, 'w') as f:
        json.dump(meta, f, indent=2, default=str)

    ind = auprc_rel.get(('drug', 'indication', 'disease'), np.nan)
    con = auprc_rel.get(('drug', 'contraindication', 'disease'), np.nan)
    print('[done] %s_seed%d | train %.1f min | edge-level indication %.4f | contraindication %.4f'
          % (arm_name, seed, (pre_s + fin_s) / 60, ind, con))

    del m, d, auroc_rel, auprc_rel
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return el

In [ ]:
if RUN_GRID:
    todo = [(a['arm'], s) for s in SEEDS for a in ARMS if not is_done(a['arm'], s)]
    if STOP_AFTER_N_RUNS:
        todo = todo[:STOP_AFTER_N_RUNS]
    print('%d run(s) pending of %d total.' % (len(todo), len(ARMS) * len(SEEDS)))
    grid_t0 = time.time()
    for i, (arm_name, seed) in enumerate(todo, 1):
        print('\n>>> %d/%d  %s_seed%d  (elapsed %.1f h)'
              % (i, len(todo), arm_name, seed, (time.time() - grid_t0) / 3600))
        run_one(arm_name, seed)
    print('\nGrid finished %d run(s) in %.1f h.' % (len(todo), (time.time() - grid_t0) / 3600))
else:
    print('RUN_GRID is False - no training. Set it in section 2.')

---
## 5 - Aggregate both protocols side by side

In [ ]:
def _load_edge_level(root):
    recs = []
    for meta_p in sorted(Path(root).glob('*/run_meta.json')):
        el_p = meta_p.parent / 'edge_level_test_metrics.csv'
        if not el_p.exists():
            continue
        meta = json.loads(meta_p.read_text())
        el = pd.read_csv(el_p).set_index('relation')
        for rel, row in el.iterrows():
            recs.append({'arm': meta['arm'], 'seed': meta['seed'], 'relation': rel,
                         'AUPRC': row['AUPRC'], 'AUROC': row['AUROC'],
                         'ranking_error': meta.get('ranking_error')})
    return pd.DataFrame(recs)


def _load_ranking(root):
    recs = []
    for meta_p in sorted(Path(root).glob('*/run_meta.json')):
        s_p = meta_p.parent / 'disease_centric_summary.csv'
        if not s_p.exists():
            continue
        meta = json.loads(meta_p.read_text())
        if meta.get('ranking_error'):
            continue
        s = pd.read_csv(s_p)
        s['arm'] = meta['arm']
        s['seed'] = meta['seed']
        recs.append(s)
    return pd.concat(recs, ignore_index=True) if recs else pd.DataFrame()


def _ci95(x):
    x = np.asarray(x, dtype=float)
    return 1.96 * x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan


ROW = {'indication': 'drug --indication--> disease',
       'contraindication': 'drug --contraindication--> disease',
       'micro': 'MICRO (drug-disease)'}

edge_long = _load_edge_level(ABL_ROOT)
rank_long = _load_ranking(ABL_ROOT)
edge_summary = rank_summary_all = None

if edge_long.empty:
    print('No runs on disk yet - run section 4 first.')
else:
    wide = (edge_long[edge_long.relation.isin(ROW.values())]
            .pivot_table(index=['arm', 'seed'], columns='relation', values='AUPRC')
            .rename(columns={v: k for k, v in ROW.items()}))
    ref = wide.xs(REF_ARM, level='arm') if REF_ARM in set(wide.index.get_level_values('arm')) else None

    out = []
    for arm, g in wide.groupby(level='arm'):
        g = g.droplevel('arm')
        rec = {'arm': arm, 'n_seeds': len(g)}
        for key in ROW:
            rec['%s_AUPRC' % key] = g[key].mean()
            if arm != REF_ARM and ref is not None:
                paired = (g[key] - ref[key]).dropna()
                rec['%s_delta' % key] = paired.mean() if len(paired) else np.nan
                rec['%s_delta_CI' % key] = _ci95(paired)
                rec['%s_hurt' % key] = int((paired < 0).sum())
            elif arm == REF_ARM:
                rec['%s_delta' % key] = 0.0
        out.append(rec)
    edge_summary = pd.DataFrame(out)
    print('EDGE-LEVEL (chance = 0.500, one sampled negative per positive)\n')
    display(edge_summary.style.format('{:.4f}', subset=[
        c for c in edge_summary.columns if c.endswith(('AUPRC', 'delta', 'CI'))],
        na_rep='-').hide(axis='index'))

    bad = edge_long[edge_long.ranking_error.notna()][['arm', 'seed', 'ranking_error']].drop_duplicates()
    if len(bad):
        print('\n[ranking failed on %d run(s)] - fix with rerank(arm, seed):' % len(bad))
        display(bad.style.hide(axis='index'))

if rank_long.empty:
    print('\nNo ranking results yet.')
else:
    # mean over diseases within a run is already done; now pair across seeds
    piv = rank_long.pivot_table(index=['relation', 'metric', 'seed'], columns='arm', values='mean')
    rows = []
    for (rel, met), g in piv.groupby(level=['relation', 'metric']):
        g = g.droplevel(['relation', 'metric'])
        if REF_ARM not in g.columns:
            continue
        for arm in [c for c in g.columns if c != REF_ARM]:
            paired = (g[arm] - g[REF_ARM]).dropna()
            if not len(paired):
                continue
            rows.append({'relation': rel, 'metric': met, 'arm': arm,
                         REF_ARM: g[REF_ARM].mean(), arm: g[arm].mean(),
                         'delta': paired.mean(), 'CI95': _ci95(paired),
                         'hurt': int((paired < 0).sum()), 'n': len(paired),
                         'rel_change_%': (100 * paired.mean() / g[REF_ARM].mean()
                                          if g[REF_ARM].mean() else np.nan)})
    rank_summary_all = pd.DataFrame(rows)
    rank_summary_all.to_csv(ABL_ROOT / 'ranking_paired_summary.csv', index=False)

    print('\nDISEASE-CENTRIC RANKING (the paper protocol) - headline metrics\n')
    head = rank_summary_all[rank_summary_all.metric.isin(RANKING_HEADLINE)].copy()
    head['_m'] = head.metric.map({m: i for i, m in enumerate(RANKING_HEADLINE)})
    head = head.sort_values(['relation', '_m']).drop(columns='_m')
    show = [c for c in ['relation', 'metric', REF_ARM, 'only_disease_relations', 'delta', 'CI95',
                        'rel_change_%', 'hurt', 'n'] if c in head.columns]
    display(head[show].style.format('{:.4f}', subset=[
        c for c in show if c in (REF_ARM, 'only_disease_relations', 'delta', 'CI95')], na_rep='-')
        .format({'rel_change_%': '{:+.1f}'}, na_rep='-').hide(axis='index'))
    print('-> %s' % (ABL_ROOT / 'ranking_paired_summary.csv').relative_to(REPO_ROOT))

### 5.1 - Does the headline survive?

The question in one number: under edge-level AUPRC the keep-only arm retains ~80% of the above-chance
indication margin. Under disease-centric ranking there is no 0.500 floor to normalise against, so the
comparable statistic is the **relative** change in a recall-at-k metric. If `Recall@5%` for indications
falls by a few percent relative, the headline survives the harsher protocol. If it halves, the headline
is specific to the lenient metric and the paper draft must say so.

In [ ]:
KEY_METRIC, KEY_REL = 'Recall@5%', 'rev_indication'

if rank_summary_all is None or rank_summary_all.empty:
    print('Need ranking results for both arms.')
else:
    sel = rank_summary_all[(rank_summary_all.metric == KEY_METRIC)
                           & (rank_summary_all.relation == KEY_REL)]
    if sel.empty:
        print('%s / %s not present. Available:' % (KEY_REL, KEY_METRIC))
        print(sorted(rank_summary_all.relation.unique()))
        print(sorted(rank_summary_all.metric.unique())[:30])
    else:
        r = sel.iloc[0]
        print('=' * 78)
        print('%s, %s' % (KEY_REL, KEY_METRIC))
        print('  full_kg                %.4f' % r[REF_ARM])
        print('  only_disease_relations %.4f' % r['only_disease_relations'])
        print('  paired delta           %+.4f +/- %.4f  (%+.1f%% relative, hurt %d/%d)'
              % (r['delta'], r['CI95'], r['rel_change_%'], r['hurt'], r['n']))
        print('=' * 78)

        if edge_summary is not None and 'indication_delta' in edge_summary.columns:
            e = edge_summary.set_index('arm')
            ed = e.loc['only_disease_relations', 'indication_delta']
            er = 100 * ed / (e.loc[REF_ARM, 'indication_AUPRC'] - 0.5)
            print('\nedge-level, same runs: %+.4f indication AUPRC (%.1f%% of the above-chance margin)'
                  % (ed, er))

        rc = r['rel_change_%']
        print('\nverdict:')
        if rc > -15:
            print('  SURVIVES. The keep-only result is not an artefact of the lenient metric; the')
            print('  9-edge-type graph is within %.0f%% of the full graph under the paper protocol too.'
                  % abs(rc))
        elif rc < -40:
            print('  DOES NOT SURVIVE. The 9-edge-type graph loses %.0f%% of ranking performance.'
                  % abs(rc))
            print('  Restrict the claim to edge-level AUPRC explicitly, or drop it as a headline.')
        else:
            print('  WEAKENED (%.0f%% relative). Report both protocols; the 80%%-of-headroom framing')
            print('  is metric-specific and should not be the headline on its own.' % abs(rc))

        print('\nAlso check the other two relations and the other k values above - a result that holds')
        print('at Recall@10%% but not Recall@10 is a result about breadth, not about top-of-list quality.')

---
## 6 - Notes for the write-up

Whatever the outcome, this notebook converts the paper draft's pre-submission item

> Reproduce the keep-only headline under the paper's full-ranking protocol

from "to do" into a reported result, and it supplies the sentence that a *Matters Arising* reviewer will
otherwise demand.

### Caveats that remain

1. **This is the library's ranking implementation, not the paper's exact script.** It is the right
   choice - it is the authors' own code - but the paper's reported figures also involve their own
   post-processing, so quote *relative* changes between arms here, never absolute agreement with the
   published numbers. The same rule already applies to every other grid.
2. **`Recall@k%` depends on the candidate pool.** `calculate_metrics` sets `k` from the number of
   drugs/diseases carrying the relation (`subset_mode=True`), so the pool is relation-specific. Compare
   arms, not relations.
3. **Only two arms.** If the ranking protocol changes the picture, the encoder arms (`light_gnn`,
   `mlp`) and `proto_off` deserve the same treatment before anything in the paper is reworded. Each is
   another ~10 training runs.
4. **10 seeds, not 20.** Extend `SEEDS` and re-run; finished runs are skipped.